# 04 · Activation Functions

**Learning objectives**
1. Know the formula, range, and derivative of step, sigmoid, tanh, ReLU, Leaky ReLU and softmax.
2. Explain why a derivative that is too small stops learning (saturation).
3. Explain why softmax needs a stability trick.

Modules: `src/activations.py` (tested in `tests/test_activations_losses.py`).

## 1. Theory

An activation $f$ turns the weighted sum $z$ into the output. Without a nonlinear $f$, stacked layers collapse to one linear map (notebook 05).

| Function | Formula | Range | Derivative |
|---|---|---|---|
| Step | $1$ if $z\ge 0$, else $0$ | $\{0,1\}$ | $0$ almost everywhere |
| Sigmoid | $\1(z)=\frac{1}{1+e^{-z}}$ | $(0,1)$ | $\1(1-\1)$, max $0.25$ |
| Tanh | $\1 z$ | $(-1,1)$ | $1-\1^2 z$, max $1$ |
| ReLU | $\max(0,z)$ | $[0,\infty)$ | $1$ if $z>0$, else $0$ |
| Leaky ReLU | $z$ if $z\ge0$, else $\alpha z$ | $\mathbb{R}$ | $1$ or $\alpha$ |
| Softmax | $\frac{e^{z_i}}{\sum_k e^{z_k}}$ | $(0,1)$, sums to 1 | $s_i(\delta_{ij}-s_j)$ |

**Why the derivative matters.** Training multiplies derivatives across layers (notebook 08). A factor at most $0.25$ shrinks a signal every layer.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

from activations import step, sigmoid, dsigmoid, tanh, dtanh, relu, drelu, leaky_relu, dleaky_relu, softmax, naive_softmax

z = np.linspace(-5, 5, 401)
fig, axes = plt.subplots(2, 3, figsize=(13, 7.5))
panels = [("Step", step, None), ("Sigmoid", sigmoid, dsigmoid), ("Tanh", tanh, dtanh),
          ("ReLU", relu, drelu), ("Leaky ReLU (alpha=0.01)", leaky_relu, dleaky_relu)]
for ax, (name, f, df) in zip(axes.flat, panels):
    ax.plot(z, f(z), label="f(z)")
    if df is not None:
        ax.plot(z, df(z), "--", label="f'(z)")
    ax.set_title(name); ax.grid(True); ax.legend(fontsize=8); ax.set_xlabel("z")
ax = axes.flat[5]
s = softmax(np.array([2.0, 1.0, 0.1]))
ax.bar(["z1=2", "z2=1", "z3=0.1"], s)
for i, v in enumerate(s):
    ax.text(i, v + 0.02, f"{v:.4f}", ha="center")
ax.set_ylim(0, 1); ax.set_title("Softmax([2, 1, 0.1])"); ax.grid(True, axis="y")
plt.tight_layout(); plt.show()

## 2. Numerical checks

Values from the formulas, compared with the module.

In [2]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

print("sigmoid(0)  =", float(sigmoid(np.array(0.0))), "  dsigmoid(0) =", float(dsigmoid(np.array(0.0))))
print("max dsigmoid on grid =", float(dsigmoid(z).max()))
print("tanh(1)  =", round(float(tanh(np.array(1.0))), 4), "  dtanh(1) =", round(float(dtanh(np.array(1.0))), 4))
print("relu([-2, 3]) =", relu(np.array([-2.0, 3.0])), "  leaky(-2) =", float(leaky_relu(np.array(-2.0))))
print("softmax([2,1,0.1]) =", softmax(np.array([2.0, 1.0, 0.1])).round(4))

sigmoid(0)  = 0.5   dsigmoid(0) = 0.25
max dsigmoid on grid = 0.25
tanh(1)  = 0.7616   dtanh(1) = 0.42
relu([-2, 3]) = [0. 3.]   leaky(-2) = -0.02
softmax([2,1,0.1]) = [0.659  0.2424 0.0986]


## 3. Saturation: why small derivatives stop learning

A neuron whose input is far from zero has a tiny sigmoid derivative. Multiply one factor per layer and the gradient shrinks geometrically.

In [3]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

for L in [1, 3, 5, 10]:
    print(f"{L:2d} sigmoid layers, best-case gradient factor <= {0.25**L:.3e}")
print("sigmoid'(5)  =", float(dsigmoid(np.array(5.0))))

 1 sigmoid layers, best-case gradient factor <= 2.500e-01
 3 sigmoid layers, best-case gradient factor <= 1.562e-02
 5 sigmoid layers, best-case gradient factor <= 9.766e-04
10 sigmoid layers, best-case gradient factor <= 9.537e-07
sigmoid'(5)  = 0.006648056670790033


## 4. Stability: softmax with large logits

Computing $e^{z}$ directly overflows for large $z$. Subtracting the maximum gives the same result.

In [4]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

big = np.array([1002.0, 1001.0, 1000.0])
with np.errstate(over="ignore", invalid="ignore"):
    print("naive softmax:", naive_softmax(big))
print("stable softmax:", softmax(big).round(4))

naive softmax: [nan nan nan]
stable softmax: [0.6652 0.2447 0.09  ]


## 5. Interpretation

- Step has no useful derivative for gradient training, which is why the perceptron uses the error directly (notebook 02).
- Sigmoid and tanh saturate, so their derivatives can be small. ReLU keeps derivative 1 for active units but can be 0 for all inputs of a dead unit.
- Softmax is the standard output for one-of-K classes, and it needs the max shift to be stable.

## 6. Conclusion

The activation sets both the forward output and the gradient that training can use. The choice is part of the architecture.

## 7. Exercises

1. Find the input $z$ where $\\1'(z)=0.1$. Check numerically.
2. Show that $\\1(z)=2\\1(2z)-1$ for three values of $z$.
3. For softmax, what happens to the outputs if all logits increase by 100? Explain with the formula.